In [0]:
# Configurações
dbutils.widgets.dropdown(
    "camada",
    "bronze",
    ["bronze", "silver", "silver", "gold"]
)

camada = dbutils.widgets.get("camada")

catalogo = "bikestore"

print(f"Validando camada: {camada}")

In [0]:
# Listagem de tabelas
schema = f"`{catalogo}`.`{camada}`"

tabelas = spark.sql(f"SHOW TABLES IN {schema}")

display(tabelas)

In [0]:
# Validações das tabelas
resultados = []

nomes_tabelas = [row.tableName for row in tabelas.collect() if not row.isTemporary]

if not nomes_tabelas:
    raise Exception(f"Nenhuma tabela encontrada em {schema}")

for nome in nomes_tabelas:
    tabela = f"{schema}.`{nome}`"

    try:
        df = spark.table(tabela)

        possui_dados = df.limit(1).count() > 0
        possui_colunas = len(df.columns) > 0

        status = "APROVADO" if possui_dados and possui_colunas else "REPROVADO"

        resultados.append(
            {
                "tabela": nome,
                "possui_dados": possui_dados,
                "possui_colunas": possui_colunas,
                "status": status,
            }
        )

    except Exception as erro:
        resultados.append(
            {
                "tabela": nome,
                "possui_dados": False,
                "possui_colunas": False,
                "status": "ERRO",
            }
        )

        print(f"Erro na tabela {nome}: {erro}")

df_validacao = spark.createDataFrame(resultados)

display(df_validacao)

In [0]:
# Aprovar ou reprovar Task
reprovadas = df_validacao.filter(
    "status != 'APROVADO'"
).count()

if reprovadas > 0:
    raise Exception(
        f"Validação reprovada! "
        f"{reprovadas} tabela(s) com problemas."
    )

print("Todas as tabelas foram aprovadas!")